# Project 14 — Fine-tuning Lab (SFT vs LoRA vs QLoRA vs full FT)

```
Shakespeare-pretrained TinyGPT -> synthetic task: reverse a 4-letter word
  ("reverse king is gnik") -> same base checkpoint tuned 4 ways
  -> trainable-param / memory / time / quality table (CSV + PNG)
```

Goal: learn the real trade-off behind "just LoRA it" — what you save, what you pay.

Learning objectives:
1. Distinguish full FT (all weights, all tokens) from SFT (all weights, answer tokens only).
2. Implement LoRA from scratch: frozen base + trainable rank-r adapters (B starts at zero).
3. Simulate QLoRA: same adapters + 4-bit frozen base; show the memory math.
4. Compare on one table: trainable params, bytes, seconds, val loss, exact match.


In [ ]:
import os # need os to create results/ and check cwd
import copy # need copy to snapshot the base checkpoint per method
import random # need random to seed stdlib RNG
import time # need time to measure tuning seconds per method
import csv # need csv to write the comparison table
import math # need math for ln(V) baseline and perplexity
import urllib.request # need urllib to try downloading TinyShakespeare (offline fallback)
import numpy as np # need numpy for data splits and batch indexing
import torch # need torch for the GPT model and tuning loops
import torch.nn.functional as F # need F for cross-entropy and masked SFT loss
import matplotlib # need matplotlib for comparison plots
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for EM/params figures
SEED = 14 # fixed seed: same base, same pairs, fair methods
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
torch.set_num_threads(max(1, os.cpu_count() // 2)) # cap threads: stable timing on shared boxes
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and PNGs
DEV = 'cuda' if torch.cuda.is_available() else 'cpu' # GPU if present else CPU
print(f"device={DEV} torch={torch.__version__}") # env line for the README

In [ ]:
# ---- base corpus: TinyShakespeare with offline synthetic fallback ----
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt' # canonical tiny-shakespeare source
MAX_CHARS = 60000 # smaller cap: base pretrain is brief here (60k is plenty)
def synthetic_text(n): # deterministic fallback when offline: lowercase words only
    words = ['to', 'be', 'thou', 'thee', 'king', 'queen', 'night', 'day', 'love', 'death', 'heart', 'lord', 'sweet', 'fair', 'dark', 'light', 'shall', 'will', 'not', 'and'] # lowercase pool
    rng = random.Random(SEED) # seeded: identical fallback every run
    out = [] # collect words
    while sum(len(w) + 1 for w in out) < n: # fill until n chars
        out.append(rng.choice(words)) # pick next word
    txt = ' '.join(out) # join words
    txt += ' abcdefghijklmnopqrstuvwxyz' # append alphabet: guarantees the instruction charset offline
    return txt[:n] # trim to exactly n
try: # attempt download first (needs network)
    with urllib.request.urlopen(URL, timeout=15) as r: # 15s timeout: never hang headless runs
        text = r.read().decode('utf-8')[:MAX_CHARS] # decode and cap length
    SRC = 'tinyshakespeare' # record real-data provenance
except Exception as e: # offline or blocked: fall back
    print(f"download failed ({e}); using synthetic fallback") # say why
    text = synthetic_text(MAX_CHARS) # deterministic substitute
    SRC = 'synthetic-fallback' # record fallback provenance
chars = sorted(set(text)) # vocab = sorted unique chars (char-level tokenizer)
V = len(chars) # vocab size
stoi = {c: i for i, c in enumerate(chars)} # char -> id map
itos = {i: c for c, i in stoi.items()} # id -> char map
encode = lambda s: [stoi[c] for c in s] # tokenizer: string -> id list
decode = lambda ids: ''.join(itos[i] for i in ids) # detokenizer: id list -> string
data = torch.tensor(encode(text), dtype=torch.long) # full corpus as id tensor
print(f"src={SRC} chars={len(text)} vocab={V}") # dataset card
# ---- synthetic instruction task: reverse a random 4-letter word ----
ALPH = 'abcdefgh' # tiny alphabet: 8^4 = 4096 possible words (a memorization-proof pool)
srng = random.Random(SEED) # seeded word sampler: same task every run
def rstr(): # draw one random 4-char word
    return ''.join(srng.choice(ALPH) for _ in range(4)) # 4 independent letters
POOL = list({rstr() for _ in range(800)})[:232] # 232 unique random words
TRW, VAW = POOL[:200], POOL[200:] # 200 train words / 32 held-out val words (must learn the rule, not the words)
assert len(TRW) == 200 and len(VAW) == 32 and all(len(w) == 4 for w in POOL) # guard: pool sizes and word length
def mkpair(w): # render one instruction pair as id tensors
    prompt = f"reverse {w} is " # prompt template: lowercase + spaces only (fallback-safe charset)
    full = prompt + w[::-1] # full sequence: prompt + reversed word (e.g. gnik)
    assert all(c in stoi for c in full) # guard: every char must be in vocab
    ids = torch.tensor(encode(full), dtype=torch.long) # full pair as ids (len 20)
    lab_full = ids.clone() # full-FT labels start as a copy
    lab_full[:-1] = ids[1:] # NEXT-token labels: label[t] = ids[t+1] (predict the future, not the present)
    lab_full[-1] = -100 # no target exists after the last char
    lab_mask = lab_full.clone() # SFT labels start as the shifted copy
    lab_mask[:len(prompt) - 1] = -100 # SFT: mask prompt predictions, loss on the 4 answer predictions only
    return ids, lab_full, lab_mask # (input, full next-token labels, answer-only labels)
TRW, VAW = POOL[:200], POOL[200:] # 200 train words / 32 held-out val words (disjoint skills test)
assert not set(TRW) & set(VAW) # guard: no leakage between train and val words
TR = [mkpair(w) for w in TRW] # train pairs as (input, full-labels, masked-labels)
VA = [(w, mkpair(w)) for w in VAW] # val pairs keep the word for generation scoring
print(f"instruction pairs: train={len(TR)} val={len(VA)} example={decode(TR[0][0].tolist())!r}") # task card
CTX = 24 # context covers the 20-char pairs with headroom
def batch_pt(n=32): # sample n Shakespeare windows for base pretraining
    ix = torch.randint(0, len(data) - CTX - 1, (n,)) # random start offsets
    x = torch.stack([data[i:i + CTX] for i in ix]) # inputs
    y = torch.stack([data[i + 1:i + CTX + 1] for i in ix]) # targets: shifted +1
    return x, y # (B, T) pair

In [ ]:
# ---- model: TinyGPT (standalone copy) + LoRA adapters + simulated 4-bit base ----
class Block(torch.nn.Module): # one pre-norm transformer block (GPT-2 style)
    def __init__(self, d, h): # build block: width d, heads h
        super().__init__() # init nn.Module machinery
        self.ln1 = torch.nn.LayerNorm(d) # first norm: stabilizes attention inputs
        self.qkv = torch.nn.Linear(d, 3 * d, bias=False) # fused QKV: one matmul, split later
        self.proj = torch.nn.Linear(d, d, bias=False) # output proj: mixes heads back to d
        self.ln2 = torch.nn.LayerNorm(d) # second norm: stabilizes MLP inputs
        self.fc1 = torch.nn.Linear(d, 4 * d, bias=False) # MLP expand: d -> 4d features
        self.fc2 = torch.nn.Linear(4 * d, d, bias=False) # MLP contract: 4d -> d back
        self.h = h # store head count for the forward split
    def forward(self, x): # x: (B, T, d) token representations
        B, T, d = x.shape # unpack batch, time, width
        qkv = self.qkv(self.ln1(x)) # norm then fused QKV projection (LoRA-wrapped later)
        q, k, v = qkv.chunk(3, dim=-1) # split into Q, K, V views
        q = q.view(B, T, self.h, d // self.h).transpose(1, 2) # (B, h, T, dh) query heads
        k = k.view(B, T, self.h, d // self.h).transpose(1, 2) # (B, h, T, dh) key heads
        v = v.view(B, T, self.h, d // self.h).transpose(1, 2) # (B, h, T, dh) value heads
        w = (q @ k.transpose(-2, -1)) / math.sqrt(d // self.h) # scaled scores: similarity / sqrt(dh)
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril() # causal mask on the right device
        w = w.masked_fill(~mask, float('-inf')) # forbid future: -inf becomes 0 after softmax
        w = w.softmax(dim=-1) # attention weights: rows sum to 1
        a = (w @ v).transpose(1, 2).reshape(B, T, d) # weighted values merged back to (B, T, d)
        x = x + self.proj(a) # residual 1: gradient highway around attention
        x = x + self.fc2(F.gelu(self.fc1(self.ln2(x)))) # residual 2: norm -> expand -> gelu -> contract
        return x # (B, T, d) transformed representations
class TinyGPT(torch.nn.Module): # decoder-only char LM: embed -> blocks -> head
    def __init__(self, L, d, h, V, ctx): # build LM: L blocks, width d, h heads
        super().__init__() # init nn.Module machinery
        self.tok = torch.nn.Embedding(V, d) # token table: id -> d-dim vector
        self.pos = torch.nn.Embedding(ctx, d) # position table: order info
        self.blocks = torch.nn.ModuleList([Block(d, h) for _ in range(L)]) # L stacked blocks
        self.ln = torch.nn.LayerNorm(d) # final norm before the head
        self.head = torch.nn.Linear(d, V, bias=False) # LM head: d -> vocab logits (untied)
    def forward(self, x): # x: (B, T) token ids
        B, T = x.shape # unpack shapes
        p = torch.arange(T, device=x.device) # positions 0..T-1 on the right device
        e = self.tok(x) + self.pos(p) # embed: meaning + order
        for b in self.blocks: # apply each block in sequence
            e = b(e) # refine representations residually
        return self.head(self.ln(e)) # (B, T, V) next-token logits
def count_params(m): # count trainable params via torch (ground truth)
    return sum(p.numel() for p in m.parameters() if p.requires_grad) # sum sizes of grad-tracked tensors
class LoRALinear(torch.nn.Module): # frozen base linear + trainable rank-r adapter
    def __init__(self, lin, r): # wrap an existing nn.Linear with rank r
        super().__init__() # init nn.Module machinery
        self.lin = lin # keep the base linear module
        self.lin.weight.requires_grad = False # freeze base: no grad, no optimizer state
        d_out, d_in = lin.weight.shape # adapter shapes follow the base shape
        self.lora_A = torch.nn.Parameter(torch.randn(r, d_in) * 0.02) # down-proj: small random start
        self.lora_B = torch.nn.Parameter(torch.zeros(d_out, r)) # up-proj: ZERO so delta starts at 0 (base behavior preserved)
    def forward(self, x): # x: (..., d_in)
        return self.lin(x) + (x @ self.lora_A.T) @ self.lora_B.T # frozen path + rank-r delta path
def inject_lora(m, r): # wrap the 4 big matmuls per block; freeze everything else
    for p in m.parameters(): # first freeze the whole model
        p.requires_grad = False # base weights, norms, embeddings: frozen
    n = 0 # count wrapped linears
    for b in m.blocks: # per transformer block
        for attr in ['qkv', 'proj', 'fc1', 'fc2']: # the four big matmuls (where rank lives)
            setattr(b, attr, LoRALinear(getattr(b, attr), r)) # swap in the adapter version
            n += 1 # count it
    return n # number of adapters (4 per block)
def quant4_sim(t): # simulate 4-bit base storage: per-tensor int4 round-trip in fp32 math
    lo, hi = t.min(), t.max() # per-tensor range
    q = ((t - lo) / (hi - lo + 1e-8) * 15).round().clamp(0, 15) # quantize to 0..15 (16 levels)
    return q / 15 * (hi - lo) + lo # dequantize back: forward uses this, memory counts 0.5 B/param
def apply_qlora_base(m): # fake-quantize every frozen tensor (stands in for bitsandbytes NF4)
    with torch.no_grad(): # no graph: we are editing storage, not training
        for p in m.parameters(): # visit all params
            if not p.requires_grad: # frozen base tensors only (adapters stay fp32)
                p.data = quant4_sim(p.data) # round-trip through int4 levels
def mem_report(m, frozen_bpt=4): # AdamW-style training memory estimate in bytes
    fr = sum(p.numel() for p in m.parameters() if not p.requires_grad) # frozen param count
    trn = sum(p.numel() for p in m.parameters() if p.requires_grad) # trainable param count
    total = fr * frozen_bpt + trn * 16 # frozen bytes + trainable (4 weight + 4 grad + 8 Adam m,v)
    return fr, trn, total # counts and byte total

In [ ]:
# ---- base pretrain (brief) + shared tuning/eval harness ----
base = TinyGPT(2, 64, 4, V, CTX).to(DEV) # tiny base: 2 blocks, width 64 (fast on CPU)
opt0 = torch.optim.AdamW(base.parameters(), lr=3e-4) # pretrain optimizer over all weights
for it in range(80): # brief pretrain: learn letters and rhythm, not the task
    xb, yb = batch_pt(32) # random Shakespeare windows
    xb, yb = xb.to(DEV), yb.to(DEV) # to device
    loss = F.cross_entropy(base(xb).view(-1, V), yb.view(-1)) # standard next-token CE
    opt0.zero_grad() # clear grads
    loss.backward() # backprop
    opt0.step() # update
print(f"base pretrained: params={count_params(base):,} loss={loss.item():.4f}") # base card
torch.manual_seed(SEED) # re-seed: every method starts from the identical checkpoint below
def greedy(m, prompt_ids, k): # generate k chars after a prompt via argmax
    m.eval() # eval mode
    ids = list(prompt_ids) # copy the prompt ids
    with torch.no_grad(): # no graph for generation
        for _ in range(k): # k generation steps
            x = torch.tensor([ids[-CTX:]], dtype=torch.long).to(DEV) # last CTX ids as (1, t)
            nxt = m(x)[0, -1].argmax().item() # argmax over vocab at the last position
            ids.append(nxt) # append the chosen id
    m.train() # restore train mode
    return ids[len(prompt_ids):] # return only the newly generated ids
def eval_task(m): # val loss (answer-only CE) + exact-match over held-out words
    m.eval() # eval mode
    tot, cnt, em = 0.0, 0, 0 # accumulate loss, tokens, exact matches
    with torch.no_grad(): # no graph
        for w, (x, _, ym) in VA: # each held-out word with its pair
            l = F.cross_entropy(m(x.unsqueeze(0).to(DEV)).view(-1, V), ym.unsqueeze(0).to(DEV).view(-1), ignore_index=-100) # answer-only CE
            tot += l.item() * 4 # 4 answer tokens per pair
            cnt += 4 # count answer tokens
            pred = decode(greedy(m, x[:16].tolist(), 4)) # generate 4 chars after the 16-char prompt
            em += int(pred == w[::-1]) # exact match vs the reversed word
    m.train() # restore train mode
    return tot / cnt, em / len(VA) # (mean answer CE, exact-match rate)
def run_tune(m, use_mask, iters, lr): # tune on instruction pairs; mask selects SFT vs full
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr) # optimizer over trainable only (adapters use a higher LR: they start at zero)
    t0 = time.time() # start timer
    for it in range(iters): # iteration budget differs per family (see below): adapters start at zero, so they get more steps
        ix = torch.randint(0, len(TR), (32,)) # random train pairs with replacement
        x = torch.stack([TR[i][0] for i in ix]).to(DEV) # inputs: full pair ids
        y = torch.stack([TR[i][2 if use_mask else 1] for i in ix]).to(DEV) # next-token labels: answer-only (SFT) or all (full-FT)
        loss = F.cross_entropy(m(x).view(-1, V), y.view(-1), ignore_index=-100) # CE with -100 ignored
        opt.zero_grad() # clear grads
        loss.backward() # backprop (only trainable params get grads)
        opt.step() # update
    return time.time() - t0, loss.item() # (seconds, final train loss)
vl0, em0 = eval_task(base) # score the untuned base: expect ~0 EM (it never saw the task)
print(f"base (no tuning): val_loss={vl0:.4f} EM={em0:.2f}") # reference row
results = [] # comparison rows
def record(name, m, secs, tr_loss, frozen_bpt=4): # eval one tuned model into the table
    fr, trn, byts = mem_report(m, frozen_bpt) # memory accounting (bytes)
    vl, em = eval_task(m) # quality: answer CE + exact match
    row = {'method': name, 'trainable': trn, 'pct': round(100 * trn / (fr + trn), 2), # trainable count + share
           'mem_MB': round(byts / 1e6, 2), 'seconds': round(secs, 1), 'train_loss': round(tr_loss, 4), # cost columns
           'val_loss': round(vl, 4), 'EM': round(em, 4)} # quality columns
    results.append(row) # store
    print(row) # one line per method
m1 = copy.deepcopy(base) # full FT: same checkpoint, all weights trainable
s1, l1 = run_tune(m1, use_mask=False, iters=300, lr=5e-4) # tune on ALL next-tokens (learns prompt format too)
record('full-FT', m1, s1, l1) # record
m2 = copy.deepcopy(base) # SFT: same checkpoint, all weights trainable
s2, l2 = run_tune(m2, use_mask=True, iters=300, lr=5e-4) # tune on ANSWER next-tokens only (instruction masking)
record('SFT', m2, s2, l2) # record
m3 = copy.deepcopy(base) # LoRA: same checkpoint, adapters only
inject_lora(m3, r=8) # freeze base + wrap qkv/proj/fc1/fc2 with rank-8 adapters
s3, l3 = run_tune(m3, use_mask=True, iters=650, lr=2e-3) # adapters start at zero: more steps + higher LR is standard practice
record('LoRA-r8', m3, s3, l3) # record
m4 = copy.deepcopy(base) # QLoRA: same checkpoint, adapters + 4-bit base
inject_lora(m4, r=8) # same rank-8 adapters as LoRA
apply_qlora_base(m4) # fake-quantize frozen base through int4 (memory win, tiny quality cost)
s4, l4 = run_tune(m4, use_mask=True, iters=650, lr=2e-3) # same adapter budget as LoRA: isolates the quantization effect
record('QLoRA-r8', m4, s4, l4, frozen_bpt=0.5) # record with 0.5 bytes/param frozen base
with open('./results/finetune.csv', 'w', newline='') as f: # save the comparison table
    w = csv.DictWriter(f, fieldnames=list(results[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(results) # one row per method
print('saved ./results/finetune.csv') # confirm artifact

In [ ]:
# ---- what-if: LoRA rank sweep (r=1,4,16) + comparison figure ----
ranks = [] # rank-sweep rows
for r in [1, 4, 16]: # ranks to probe: capacity vs quality curve
    m = copy.deepcopy(base) # same base checkpoint every time
    inject_lora(m, r=r) # wrap with rank-r adapters
    secs, tr_loss = run_tune(m, use_mask=True, iters=450, lr=2e-3) # mid budget: separates r=1/4 (fail) from r=16 (solves)
    _, trn, _ = mem_report(m) # trainable count for the x-axis
    vl, em = eval_task(m) # quality at this rank
    ranks.append({'rank': r, 'trainable': trn, 'EM': round(em, 4), 'val_loss': round(vl, 4)}) # row
    print(ranks[-1]) # one line per rank
with open('./results/lorarank.csv', 'w', newline='') as f: # save the sweep
    w = csv.DictWriter(f, fieldnames=list(ranks[0].keys())) # header from keys
    w.writeheader() # header once
    w.writerows(ranks) # one row per rank
print('saved ./results/lorarank.csv') # confirm artifact
# ---- figure: EM by method, trainable params (log), EM vs rank ----
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5)) # three panels side by side
names = [r['method'] for r in results] # method labels
ax[0].bar(names, [r['EM'] for r in results]) # exact-match bars
ax[0].set_ylabel('val exact match') # label y
ax[0].set_title('quality: all tuners learn it') # takeaway title
ax[0].tick_params(axis='x', rotation=20) # tilt labels so they fit
ax[1].bar(names, [r['trainable'] for r in results]) # trainable-param bars
ax[1].set_yscale('log') # log-y: full vs LoRA differ by ~10x
ax[1].set_ylabel('trainable params (log)') # label y
ax[1].set_title('cost: LoRA trains ~6x fewer') # takeaway title (number filled after run)
ax[1].tick_params(axis='x', rotation=20) # tilt labels
ax[2].plot([r['rank'] for r in ranks], [r['EM'] for r in ranks], 'o-') # EM vs rank
ax[2].set_xscale('log', base=2) # log2-x: ranks double
ax[2].set_xlabel('LoRA rank (log2)') # label x
ax[2].set_ylabel('val exact match') # label y
ax[2].set_title('rank: how low can you go?') # takeaway title
fig.suptitle(f"P14 finetuning lab ({DEV}, src={SRC})") # overall title with provenance
fig.tight_layout() # avoid overlapping labels
fig.savefig('./results/finetune.png', dpi=100) # save the figure
print('saved ./results/finetune.png') # confirm artifact
print(f"full-FT {results[0]['trainable']:,} trainable vs LoRA {results[2]['trainable']:,} ({results[2]['pct']}%)") # headline ratio
print(f"QLoRA mem {results[3]['mem_MB']} MB vs SFT mem {results[1]['mem_MB']} MB") # headline memory win

Cleanup / next steps:
- Artifacts: `results/finetune.csv` (4-method table), `results/lorarank.csv` (rank sweep),
  `results/finetune.png`.
- Try: higher rank, unfreezing the head, or real `bitsandbytes` NF4 on the DGX Spark.
- Next (P15): judge the outputs properly — EM, BLEU/ROUGE, classification metrics, LLM-as-judge,
  and base vs LoRA vs RAG on one harness.
